In [4]:
import os
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Dense, Flatten, Dropout, GlobalAveragePooling2D
from tensorflow.keras.optimizers import Adam

# Configuración inicial
processed_data_dir = 'dataset_procesado'  # Carpeta con las imágenes preprocesadas
output_model_path = 'modelo_identificacion_cerdos.h5'  # Ruta donde se guardará el modelo
img_size = (224, 224)  # Tamaño de entrada requerido por MobileNet
batch_size = 16
epochs = 20

# Generador de imágenes con división para validación
data_gen = ImageDataGenerator(
    rescale=1.0/255,
    validation_split=0.2  # 80% entrenamiento, 20% validación
)

# Carga los datos desde carpetas
train_generator = data_gen.flow_from_directory(
    processed_data_dir,
    target_size=img_size,
    batch_size=batch_size,
    class_mode='categorical',
    subset='training'
)

val_generator = data_gen.flow_from_directory(
    processed_data_dir,
    target_size=img_size,
    batch_size=batch_size,
    class_mode='categorical',
    subset='validation'
)

# Cargar la base MobileNetV2 sin su capa de salida
base_model = MobileNetV2(weights='imagenet', include_top=False, input_shape=(224, 224, 3))
base_model.trainable = False  # No se entrena la base

# Agregar capas superiores personalizadas
x = base_model.output
x = GlobalAveragePooling2D()(x)
x = Dense(128, activation='relu')(x)
x = Dropout(0.5)(x)
predictions = Dense(train_generator.num_classes, activation='softmax')(x)
model = Model(inputs=base_model.input, outputs=predictions)

# Compilar modelo
model.compile(
    optimizer=Adam(learning_rate=0.0001),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

# Entrenar el modelo
model.fit(
    train_generator,
    validation_data=val_generator,
    epochs=epochs,
    steps_per_epoch=train_generator.samples // batch_size,
    validation_steps=val_generator.samples // batch_size
)

# Guardar modelo entrenado
model.save(output_model_path)
print(f"\n✅ Modelo entrenado guardado en: {output_model_path}")


Found 167 images belonging to 9 classes.
Found 38 images belonging to 9 classes.
Epoch 1/20
10/10 ━━━━━━━━━━━━━━━━━━━━ 5s 256ms/step - accuracy: 0.1463 - loss: 2.7080 - val_accuracy: 0.2188 - val_loss: 2.0449
Epoch 2/20
10/10 ━━━━━━━━━━━━━━━━━━━━ 1s 41ms/step - accuracy: 0.1250 - loss: 3.1121 - val_accuracy: 0.2812 - val_loss: 1.9658
Epoch 3/20
10/10 ━━━━━━━━━━━━━━━━━━━━ 2s 209ms/step - accuracy: 0.1459 - loss: 2.3528 - val_accuracy: 0.2500 - val_loss: 1.8480
Epoch 4/20
10/10 ━━━━━━━━━━━━━━━━━━━━ 1s 40ms/step - accuracy: 0.2500 - loss: 2.0134 - val_accuracy: 0.1875 - val_loss: 1.8472
Epoch 5/20
10/10 ━━━━━━━━━━━━━━━━━━━━ 2s 182ms/step - accuracy: 0.2335 - loss: 2.1441 - val_accuracy: 0.4062 - val_loss: 1.6885
Epoch 6/20
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 37ms/step - accuracy: 0.2500 - loss: 1.7217 - val_accuracy: 0.3750 - val_loss: 1.6577
Epoch 7/20
10/10 ━━━━━━━━━━━━━━━━━━━━ 2s 185ms/step - accuracy: 0.2895 - loss: 1.7811 - val_accuracy: 0.3750 - val_loss: 1.6373
Epoch 8/20
10/10 ━━━━━━━━━


✅ Modelo entrenado guardado en: modelo_identificacion_cerdos.h5
